In [1]:
# set up environment
import sys
import json
import re
import requests
import pandas as pd
from pathlib import Path
import json
import csv
from caveclient import CAVEclient

import urllib.parse

from pathlib import Path
#pip install neuprint-python
from neuprint import Client, fetch_neurons, fetch_custom, NeuronCriteria as NC


print("Python executable:", sys.executable)
print("Imports OK")

c:\Users\JHS\Documents\Python\cave_env\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Python executable: c:\Users\JHS\Documents\Python\cave_env\Scripts\python.exe
Imports OK


In [3]:
# set up directories
PROJECT_ROOT = Path.cwd() #anchor to current notebook location


OUTPUT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_add neuromeres T3L 20260412"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("Output:", OUTPUT_DIR)



Output: c:\Users\JHS\Documents\Python\project_folder_4B\output-MANCv1.2.3_04B_add neuromeres T3L 20260412


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

In [5]:
from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


In [9]:
#Fetch neurons to popualte the state later: one neuromere with 04B add T2 left side

from neuprint import fetch_custom

cypher = """
MATCH (n:Neuron)
WHERE n.hemilineage = '04B'
  AND n.somaNeuromere = 'T3'
  AND n.somaSide = 'LHS'
RETURN
  n.bodyId AS bodyId,
  n.type AS type,
  n.instance AS instance,
  n.class AS class,
  n.subclass AS subclass,
  n.hemilineage AS hemilineage,
  n.somaNeuromere AS somaNeuromere,
  n.somaSide AS somaSide,
  n.rootSide AS rootSide,
  n.longTract AS longTract,
  n.birthtime AS birthtime
ORDER BY bodyId
"""

df_4b_t3_lhs = fetch_custom(cypher, client=c)
df_4b_t3_lhs.head() 
df_4b_t3_lhs["bodyId"].nunique() # n=97

89

In [12]:
# Generate a new state that diaplays the 4B morphology clusters, as layers for each subclass WITH  all neurons in one layer displayed with the same color. 04BT2l

import json
import urllib.parse
from itertools import cycle


#generate new, clean state:

##define sources:
# 1) Paste a known-good MANC Clio URL here (we wont repopulate it; it is jsut for retrieving sources)
BASE_URL = "https://clio-ng.janelia.org/#!%7B%22title%22:%22manc-v1.2.3-neuprint-layers%22%2C%22dimensions%22:%7B%22x%22:%5B8e-9%2C%22m%22%5D%2C%22y%22:%5B8e-9%2C%22m%22%5D%2C%22z%22:%5B8e-9%2C%22m%22%5D%7D%2C%22position%22:%5B23056.5%2C29733.5%2C41138.5%5D%2C%22crossSectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22:1%2C%22projectionOrientation%22:%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22projectionScale%22:91364.04452716278%2C%22layers%22:%5B%7B%22type%22:%22image%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22name%22:%22em%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%5B%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2%22%2C%22subsources%22:%7B%22default%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%7B%22url%22:%22precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/%22%2C%22enableDefaultSubsources%22:false%7D%5D%2C%22toolBindings%22:%7B%22Q%22:%22selectSegments%22%7D%2C%22tab%22:%22segments%22%2C%22segments%22:%5B%2210007%22%5D%2C%22segmentColors%22:%7B%2233946%22:%22#808080%22%7D%2C%22name%22:%22manc:v1.2.3%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/all-vnc-roi%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22name%22:%22all-tissue%22%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/synaptic-neuropil%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22rendering%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%221%22%5D%2C%22segmentDefaultColor%22:%22#ffffff%22%2C%22segmentColors%22:%7B%221%22:%22#ffffff%22%7D%2C%22name%22:%22all-synaptic%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/roi-202208%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22selectedAlpha%22:0%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22neuropils%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/nerve-roi-202301%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%2C%22mesh%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22objectAlpha%22:0.1%2C%22segments%22:%5B%221%22%2C%2210%22%2C%2211%22%2C%2212%22%2C%2213%22%2C%2214%22%2C%2215%22%2C%2216%22%2C%2217%22%2C%2218%22%2C%2219%22%2C%222%22%2C%2220%22%2C%2221%22%2C%2222%22%2C%2223%22%2C%2224%22%2C%2225%22%2C%2226%22%2C%2227%22%2C%2228%22%2C%2229%22%2C%223%22%2C%2230%22%2C%2231%22%2C%2232%22%2C%2233%22%2C%2234%22%2C%2235%22%2C%224%22%2C%225%22%2C%226%22%2C%227%22%2C%228%22%2C%229%22%5D%2C%22name%22:%22nerves%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPre%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPostPartners%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPre%20&&%20showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPostPartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPre%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPostPartners%22:false%2C%22postColor%22:%22#00ffff%22%2C%22preConfidence%22:0.4%2C%22postConfidence%22:0.4%7D%2C%22linkedSegmentationLayer%22:%7B%22pre_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22pre_synaptic_cell%22%5D%2C%22name%22:%22presyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22annotation%22%2C%22source%22:%22precomputed://gs://manc-seg-v1p2/manc-v1.2-synapse-partners-minconf-0.0.precomputed%22%2C%22tab%22:%22rendering%22%2C%22ignoreNullSegmentFilter%22:false%2C%22shader%22:%22#uicontrol%20bool%20showPrePartners%20checkbox%28default=true%29%5Cn#uicontrol%20bool%20showPost%20checkbox%28default=true%29%5Cn%5Cn#uicontrol%20vec3%20preColor%20color%28default=%5C%22red%5C%22%29%5Cn#uicontrol%20vec3%20postColor%20color%28default=%5C%22blue%5C%22%29%5Cn%5Cn//%20Note:%5Cn//%20%20%20For%20the%20MANC%20in%20neuprint%2C%5Cn//%20%20%20connection%20strengths%20don%27t%5Cn//%20%20%20include%20any%20synapses%20below%5Cn//%20%20%20confidence%200.4.%5Cn#uicontrol%20float%20preConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn#uicontrol%20float%20postConfidence%20slider%28min=0%2C%20max=1%2C%20default=0.4%29%5Cn%5Cnvoid%20main%28%29%20%7B%5Cn%20%20setColor%28defaultColor%28%29%29%3B%5Cn%5Cn%20%20float%20preSize%20=%206.0%3B%5Cn%20%20float%20postSize%20=%204.0%3B%5Cn%20%20%5Cn%20%20setEndpointMarkerColor%28%5Cn%20%20%20%20vec4%28preColor%2C%200.5%29%2C%5Cn%20%20%20%20vec4%28postColor%2C%200.5%29%29%3B%5Cn%20%20%5Cn%20%20if%20%28showPrePartners%20&&%20showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%281.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPost%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%20postSize%29%3B%5Cn%20%20%7D%5Cn%20%20else%20if%20%28showPrePartners%29%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%28preSize%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20else%20%7B%5Cn%20%20%20%20setLineWidth%280.0%29%3B%5Cn%20%20%20%20setEndpointMarkerSize%280.0%2C%200.0%29%3B%5Cn%20%20%7D%5Cn%20%20%5Cn%20%20if%20%28prop_pre_synaptic_confidence%28%29%20%3C%20preConfidence%20%7C%7C%5Cn%20%20%20%20%20%20prop_post_synaptic_confidence%28%29%20%3C%20postConfidence%29%20discard%3B%5Cn%7D%5Cn%22%2C%22shaderControls%22:%7B%22showPrePartners%22:false%2C%22postColor%22:%22#00ffff%22%7D%2C%22linkedSegmentationLayer%22:%7B%22post_synaptic_cell%22:%22manc:v1.2.3%22%7D%2C%22filterBySegmentation%22:%5B%22post_synaptic_cell%22%5D%2C%22name%22:%22postsyn%22%2C%22visible%22:false%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/mask%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22pick%22:false%2C%22tab%22:%22segments%22%2C%22segmentColors%22:%7B%221%22:%22#000000%22%7D%2C%22name%22:%22voxel-classes%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://vnc-v3-seg-3d2f1c08fd4720848061f77362dc6c17/rc5_wsexp%22%2C%22subsources%22:%7B%22default%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22segments%22%2C%22name%22:%22initial-supervoxels%22%2C%22archived%22:true%7D%2C%7B%22type%22:%22segmentation%22%2C%22source%22:%7B%22url%22:%22precomputed://gs://flyem-vnc-roi-d5f392696f7a48e27f49fa1a9db5ee3b/court-et-al-systematic-manc_tracts/mesh%22%2C%22subsources%22:%7B%22default%22:true%2C%22properties%22:true%7D%2C%22enableDefaultSubsources%22:false%7D%2C%22tab%22:%22rendering%22%2C%22saturation%22:0.5%2C%22meshSilhouetteRendering%22:4%2C%22segments%22:%5B%22100%22%2C%22101%22%2C%22102%22%2C%22103%22%2C%22104%22%2C%22105%22%2C%22106%22%5D%2C%22name%22:%22court%20et%20al.%20tracts%22%2C%22archived%22:true%7D%5D%2C%22showSlices%22:false%2C%22prefetch%22:false%2C%22selectedLayer%22:%7B%22flex%22:1.49%2C%22size%22:426%2C%22visible%22:true%2C%22layer%22:%22manc:v1.2.3%22%7D%2C%22projectionBackgroundColor%22:%22#ffffff%22%2C%22layout%22:%223d%22%2C%22settingsPanel%22:%7B%22visible%22:true%7D%2C%22selection%22:%7B%22flex%22:0.51%2C%22size%22:426%2C%22visible%22:false%7D%7D"

# 2) Decode its state
encoded_state = BASE_URL.split("#!", 1)[1]
base_state = json.loads(urllib.parse.unquote(encoded_state))

# 3) Extract the actual working sources from that scene
em_source = None
seg_source = None

for L in base_state["layers"]:
    if L.get("type") == "image" and em_source is None:
        em_source = L.get("source")
    if L.get("type") == "segmentation" and seg_source is None:
        seg_source = L.get("source")

print("EM source:", em_source)
print("Seg source:", seg_source)

##generate state
new_state = {
    "title": "MANC_v1.2.3_4B_T3L",
    "dimensions": base_state["dimensions"],
    "position": base_state["position"],
    "crossSectionOrientation": base_state["crossSectionOrientation"],
    "crossSectionScale": base_state["crossSectionScale"],
    "projectionOrientation": base_state["projectionOrientation"],
    "projectionScale": base_state["projectionScale"],
    "layout": base_state.get("layout", "3d"),
    "showSlices": base_state.get("showSlices", True),
    "projectionBackgroundColor": base_state.get("projectionBackgroundColor", "#ffffff"),
    "crossSectionBackgroundColor": base_state.get("crossSectionBackgroundColor", "#ffffff"),
    "selectedLayer": base_state.get("selectedLayer"),
    "settingsPanel": base_state.get("settingsPanel", {"visible": True}),
    "layers": [
        {
            "type": "image",
            "name": "EM",
            "source": em_source,
            "visible": True
        }
    ]
}


# --- Color palette (cycles if needed) ---
COLORS = [
    "#87CEEB",  # sky blue
    "#FFA500",  # orange
    "#E34234",  # vermillion
    "#CC99FF",  # pale violet
    "#66CDAA",  # medium aquamarine
    "#FFD700",  # gold
]

color_cycle = cycle(COLORS)


# --- Helper: upsert layer (FIXED) ---
def add_segmentation_layer(state, name, body_ids, seg_source, color):
    body_ids = sorted(set(int(x) for x in body_ids))

    layer = {
        "type": "segmentation",
        "name": name,
        "source": seg_source,
        "segments": [str(x) for x in body_ids],
        "segmentColors": {str(x): color for x in body_ids},
        "visible": True,
    }

    # replace if exists
    for i, existing in enumerate(state["layers"]):
        if existing.get("name") == name:
            state["layers"][i] = layer
            return

    # otherwise append
    state["layers"].append(layer)


# --- Clean subclass labels ---
subclass_df = df_4b_t3_lhs.copy()
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"


# --- Inspect counts ---
subclass_counts = (
    subclass_df.groupby("subclass")["bodyId"]
    .nunique()
    .sort_values(ascending=False)
)
print(subclass_counts)


# --- Build layers with colors ---
for subclass_name, group in subclass_df.groupby("subclass", sort=True):
    body_ids = group["bodyId"].dropna().astype(int).tolist()
    if not body_ids:
        continue

    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)

    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)


#add more manual layers based on my cluster review. 
#use a dictionary and loop to add the layers:
##these came from notebook 4
ir_2_04B_1_ids = []
ir_2_04B_2_ids = [] 
ir_2_04B_3_ids = [] 
ir_2_04B_4_ids =[] 
ir_2_04B_5_ids = []
ir_2_04B_6_ids = []

ir_1_1_ids = []
ir_1_2_ids = []
ir_1_3_ids = []
ir_1_4_ids = []
ir_1_5_ids = []
ir_1_6_ids = []

manual_layers = {

    "ir_2_04B_1": ir_2_04B_1_ids,
    "ir_2_04B_2": ir_2_04B_2_ids,
    "ir_2_04B_3": ir_2_04B_3_ids,
    "ir_2_04B_4": ir_2_04B_4_ids,
    "ir_2_04B_5": ir_2_04B_5_ids,
    "ir_2_04B_6": ir_2_04B_6_ids,
    "ir_1_1": ir_1_1_ids,
    "ir_1_2": ir_1_2_ids,
    "ir_1_3": ir_1_3_ids,
    "ir_1_4": ir_1_4_ids,
    "ir_1_5": ir_1_5_ids,
    "ir_1_6":ir_1_6_ids,

   
}

for name, ids in manual_layers.items():
    layer_name = f"MANC_v1.2.3_{name}"
    color = next(color_cycle)  # or set manually if you want control

    add_segmentation_layer(new_state, layer_name, ids, seg_source, color)



#Add asingle layer manually
# add_segmentation_layer(
#     new_state,
#     "MANC_v1.2.3_candidate_pair",
#     [123456789, 987654321],
#     seg_source,
#     "#FF0000"
# )



# --- Debug print ---
print("State title:", new_state["title"])
print("Number of layers:", len(new_state["layers"]))

for layer in new_state["layers"]:
    print(layer["name"], len(layer.get("segments", [])))


# --- Save ---
encoded = urllib.parse.quote(json.dumps(new_state, separators=(",", ":")))
NEW_URL = "https://clio-ng.janelia.org/#!" + encoded


STATE_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_04BT3L.json"
URL_OUT = OUTPUT_DIR / "MANC_v1.2.3_4B_04BT3L_url.txt"

with open(STATE_OUT, "w", encoding="utf-8") as f:
    json.dump(new_state, f, indent=2)

with open(URL_OUT, "w", encoding="utf-8") as f:
    f.write(NEW_URL)

print("\nSaved new state to:", STATE_OUT.resolve())
print("Saved URL to:", URL_OUT.resolve())
print("\nNew URL:\n")
print(NEW_URL)

EM source: {'url': 'precomputed://gs://flyem-vnc-2-26-213dba213ef26e094c16c860ae7f4be0/v3_emdata_clahe_xy/jpeg', 'subsources': {'default': True}, 'enableDefaultSubsources': False}
Seg source: [{'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2', 'subsources': {'default': True, 'mesh': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/type_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/segment_properties_v1.2.3/tags_property/', 'subsources': {'default': True}, 'enableDefaultSubsources': False}, {'url': 'precomputed://gs://manc-seg-v1p2/manc-seg-v1.2/numeric_properties/', 'enableDefaultSubsources': False}]
subclass
IR    59
BR    26
IA     2
II     2
Name: bodyId, dtype: int64
State title: MANC_v1.2.3_4B_T3L
Number of layers: 17
EM 0
MANC_v1.2.3_BR 26
MANC_v1.2.3_IA 2
MANC_v1.2.3_II 2
MANC_v1.2.3_IR 59
MANC_v1.2.3_ir_2_04B

In [13]:
# split out the T3L neurons:

subclass_df = df_4b_t3_lhs.copy()

#retrieve bodyIDs per subclass:
subclass_df["subclass"] = (
    subclass_df["subclass"]
    .fillna("unclassified")
    .astype(str)
    .str.strip()
)
subclass_df.loc[subclass_df["subclass"] == "", "subclass"] = "unclassified"

subclass_to_ids = (
    subclass_df.groupby("subclass")["bodyId"]
    .apply(lambda s: sorted(set(s.dropna().astype(int))))
    .to_dict()
)

##inspect them
for subclass, ids in sorted(subclass_to_ids.items()):
    print(f"{subclass}: {len(ids)} cells")
    print(ids)
    print()

# build layers
for subclass_name, body_ids in sorted(subclass_to_ids.items()):
    layer_name = f"MANC_v1.2.3_{subclass_name}"
    color = next(color_cycle)
    add_segmentation_layer(new_state, layer_name, body_ids, seg_source, color)

BR: 26 cells
[11517, 14612, 14744, 15231, 15968, 15999, 16015, 16136, 16243, 16613, 16847, 17673, 17767, 17921, 18045, 18068, 18125, 19089, 19501, 19621, 21179, 21861, 22420, 32370, 101877, 156679]

IA: 2 cells
[13043, 17924]

II: 2 cells
[10387, 11350]

IR: 59 cells
[12110, 12875, 13909, 14217, 14353, 16339, 16441, 16619, 16840, 16917, 17101, 17601, 17755, 17802, 18138, 18330, 18698, 18900, 19152, 19161, 19214, 19557, 19772, 19946, 20236, 20412, 20420, 20982, 20990, 21021, 21183, 21197, 21339, 22396, 22433, 23319, 23500, 24282, 25045, 25877, 25951, 26423, 26556, 27029, 27803, 27882, 28943, 30468, 30530, 30805, 34713, 100598, 101398, 101578, 152618, 155758, 159685, 164298, 171352]



In [14]:
#lets look at IR subclass liek I did for T1- is there dags inthe MANC description? 
#let me inspect subclass IR in 04B to see if MANC has subdivided it 

#fidn the annotations the way I see them when I select a neuron inthe API

from neuprint import fetch_neurons, NeuronCriteria as NC

criteria = NC(bodyId=df_4b_t3_lhs["bodyId"].tolist())

neurons, roi_counts = fetch_neurons(criteria, client=c)

print(neurons.columns)


#------------------------------------
## make neurons_clean should  to handle missing serialMotif values before grouping.

neurons_clean = neurons.copy()

# Flag whether a serial motif was originally annotated
neurons_clean["has_serialMotif"] = neurons_clean["serialMotif"].notna()

# Fill missing values so groupby does not drop them or leave NaN labels
neurons_clean["serialMotif"] = (
    neurons_clean["serialMotif"]
    .fillna("none")
    .astype(str)
    .str.strip()
)

# Optional cleanup for other grouping columns
# for col, default in {
#     "subclass": "unclassified",
#     "type": "unknown",
#     "instance": "none",
# }.items():
#     neurons_clean[col] = neurons_clean[col].fillna(default).astype(str).str.strip()
# table_with_ids = (
#     neurons_clean
#     .groupby(["subclass", "type", "instance", "serialMotif"])
#     .agg(
#         count=("bodyId", "nunique"),
#         bodyIds=("bodyId", lambda x: sorted(set(x)))
#     )
#     .reset_index()
# )
table = (
    neurons_clean
    .groupby(["subclass", "type", "instance", "serialMotif"])["bodyId"]
    .nunique()
    .reset_index(name="count")
)

# add flag AFTER grouping
table["has_motif"] = table["serialMotif"] != "none"

# sort
table = table.sort_values(
    by=["has_motif", "subclass", "count"],
    ascending=[False, True, False]
)

print(table)

#---------------------------------------

ir_df = neurons[neurons["subclass"] == "IR"].copy()
##temporary filtered view

ir_df_selectedcols = ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

ir_df_selectedcols

Index(['bodyId', 'instance', 'type', 'pre', 'post', 'downstream', 'upstream',
       'size', 'status', 'statusLabel', 'somaLocation', 'roiInfo',
       'hemilineage', 'ntAcetylcholineProb', 'source', 'locationType',
       'description', 'somaNeuromere', 'location', 'receptorType', 'serial',
       'entryNerve', 'synweight', 'systematicType', 'ntUnknownProb', 'origin',
       'transmission', 'cluster', 'ntGabaProb', 'tag', 'ntGlutamateProb',
       'birthtime', 'modality', 'predictedNtProb', 'somaSide', 'longTract',
       'class', 'serialMotif', 'celltypePredictedNt', 'subclassabbr',
       'subcluster', 'synonyms', 'rootSide', 'prefix', 'avgLocation',
       'exitNerve', 'vfbId', 'group', 'celltypeTotalNtPredictions', 'subclass',
       'rootLocation', 'predictedNt', 'target', 'tosomaLocation',
       'totalNtPredictions', 'inputRois', 'outputRois'],
      dtype='object')
   subclass      type       instance      serialMotif  count  has_motif
9        IA  AN04B003  AN04B003_T3_L     

,bodyId,type,instance,serialMotif,description,modality
3,12110,IN04B007,IN04B007_T3_L,None,4B in T3 LHS,None
4,12875,IN04B008,IN04B008_T3_L,independent leg,None,None
41,19161,IN04B025,IN04B025_T3_L,independent leg,20A in T3 LHS,None
17,16339,IN04B026,IN04B026_T3_L,independent leg,20A in T3 LHS,None
23,16917,IN04B031,IN04B031_T3_L,independent leg,4B in T3 LHS,None
29,17802,IN04B037,IN04B037_T3_L,independent leg,None,None
40,19152,IN04B042,IN04B042_T3_L,None,None,None
35,18138,IN04B043,IN04B043_T3_L,None,None,None
53,21021,IN04B043,IN04B043_T3_L,None,None,None
88,171352,IN04B044,IN04B044_T3_L,None,None,None


In [15]:
#ok let us make two Dfs for the 04bIR and the 20A flagged 04B IRs in T3L

#let me inspect subclass IR in 04B to see if MANC has subdivided it 

###pull all IR neurons in ir_df
ir_df = neurons[neurons["subclass"] == "IR"].copy()


ir_df[[
    "bodyId",
    "type",
    "instance",
    "serialMotif",
    "description",
    "modality"
]].sort_values("type")

ir_ids = sorted(ir_df["bodyId"].dropna().astype(int).unique().tolist())
print(ir_ids)
print("n IR neurons:", len(ir_ids))


# create flag
ir_df["maybe_20A"] = ir_df["description"].str.contains(
    "20A", case=False, na=False
)

# subset
ir_20A = ir_df[ir_df["maybe_20A"]]

ir_20A[[
    "bodyId", "type", "instance", "description", 'serialMotif'
]]

# everything else
ir_04B = ir_df[~ir_df["maybe_20A"]]



# make unique type lists for the two IR groups

types_20A = (
    ir_20A
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_04B_or_unknown = (
    ir_04B
    .groupby("type")["bodyId"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="bodyIds")
    .sort_values("type")
    .reset_index(drop=True)
)

types_20A
types_04B_or_unknown

types_20A_set = set(types_20A["type"])
types_04B_set = set(types_04B_or_unknown["type"])

overlap = types_20A_set & types_04B_set
only_20A = types_20A_set - types_04B_set
only_04B = types_04B_set - types_20A_set

print("Overlap:", overlap)
print("\nOnly in 20A:", only_20A)
print("\nOnly in 04B/unknown:", only_04B)


n_20A = ir_df[ir_df["maybe_20A"]]["bodyId"].nunique()
n_04B = ir_df[~ir_df["maybe_20A"]]["bodyId"].nunique()

print("IR maybe 20A:", n_20A)
print("IR 04B/unknown:", n_04B)

ids_20A = [int(x) for x in ir_20A["bodyId"].unique()]
ids_04B = [int(x) for x in ir_04B["bodyId"].unique()]

print(ids_20A)
print(ids_04B)



[12110, 12875, 13909, 14217, 14353, 16339, 16441, 16619, 16840, 16917, 17101, 17601, 17755, 17802, 18138, 18330, 18698, 18900, 19152, 19161, 19214, 19557, 19772, 19946, 20236, 20412, 20420, 20982, 20990, 21021, 21183, 21197, 21339, 22396, 22433, 23319, 23500, 24282, 25045, 25877, 25951, 26423, 26556, 27029, 27803, 27882, 28943, 30468, 30530, 30805, 34713, 100598, 101398, 101578, 152618, 155758, 159685, 164298, 171352]
n IR neurons: 59
Overlap: {'IN04B074', 'IN04B107'}

Only in 20A: {'IN04B104', 'IN04B095', 'IN04B113', 'IN04B026', 'IN04B025', 'IN04B112', 'IN04B114', 'IN04B092', 'IN04B110', 'IN04B105'}

Only in 04B/unknown: {'IN04B008', 'IN04B044', 'IN04B096', 'IN04B031', 'IN04B088', 'IN04B007', 'IN04B042', 'IN04B062', 'IN04B100', 'IN04B063', 'IN04B075', 'IN04B043', 'IN04B037', 'IN04B052', 'IN04B080', 'IN04B083', 'IN04B078', 'IN04B068'}
IR maybe 20A: 23
IR 04B/unknown: 36
[16339, 16619, 17601, 18698, 18900, 19161, 21339, 22396, 23500, 25045, 25877, 26423, 26556, 27029, 27803, 27882, 2894

In [ ]:
#Jelly manual cluster annotation- reviewed in neuroglancer for morphologies
ir_all = [12110, 12875, 13909, 14217, 14353, 16339, 16441, 16619, 16840, 16917, 17101, 17601, 17755, 17802, 18138, 18330, 18698, 18900, 19152, 19161, 19214, 19557, 19772, 19946, 20236, 20412, 20420, 20982, 20990, 21021, 21183, 21197, 21339, 22396, 22433, 23319, 23500, 24282, 25045, 25877, 25951, 26423, 26556, 27029, 27803, 27882, 28943, 30468, 30530, 30805, 34713, 100598, 101398, 101578, 152618, 155758, 159685, 164298, 171352]


ir_20A = [16339, 16619, 17601, 18698, 18900, 19161, 21339, 22396, 23500, 25045, 25877, 26423, 26556, 27029, 27803, 27882, 28943, 30530, 30805, 34713, 155758, 159685, 164298]


print (len(ir_all))

ir_04B = [2110, 12875, 13909, 14217, 14353, 16441, 16840, 16917, 17101, 17755, 17802, 18138, 18330, 19152, 19214, 19557, 19772, 19946, 20236, 20412, 20420, 20982, 20990, 21021, 21183, 21197, 22433, 23319, 24282, 25951, 30468, 100598, 101398, 101578, 152618, 171352]

print (len(ir_04B))

#Jelly clusters: 
ir_2_04B_1_ids = [12875, 13909, 16441, 16840, 19214, 20420, 101578, 152618 ]
ir_2_04B_2_ids = [14217, 14353, 16917, 17101, 17755, 17802, 18138, 18330, 19152, 20236, 171352]

ir_2_04B_3_ids = [20412, 20990, 22433, 24282, 100598]

ir_2_04B_4_ids =[19946, 21021, 30468] 
ir_2_04B_5_ids =[19557, 19772, 23319] 
ir_2_04B_6_ids =[2110,  20982, 21183, 21197, 25951]  #2110 does not show up
ir_2_04B_7_ids =[101398]
#ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids  and x not in ir_2_04B_3_ids  and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids]
#ir_remaining = [x for x in ir_04B if x not in ir_2_04B_1_ids and x not in ir_2_04B_2_ids and x not in ir_2_04B_4_ids and x not in ir_2_04B_5_ids]

print(ir_remaining)



# #did I miss anything: 
all_groups = (
    set(ir_2_04B_1_ids)
    | set(ir_2_04B_2_ids)
    | set(ir_2_04B_3_ids)
    | set(ir_2_04B_4_ids)
    | set(ir_2_04B_5_ids)
    | set(ir_2_04B_6_ids)
    | set(ir_2_04B_7_ids)
)


ir_set = set(ir_04B)

missing = ir_set - all_groups
extra = all_groups - ir_set

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_2_04B_1_ids
    + ir_2_04B_2_ids
    + ir_2_04B_3_ids
    + ir_2_04B_4_ids
    + ir_2_04B_5_ids
    + ir_2_04B_6_ids
    
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)



59
36
[2110, 19557, 19772, 20982, 21183, 21197, 23319, 25951, 101398]
Missing from groups: set()
Extra in groups: {29104, 15986, 26083}
Duplicates across groups: {19557: 2, 19772: 2, 23319: 2}


In [37]:
#beautiful. Now we do group T1l-IR tagged as 20B


ir_all = [13910, 14358, 14496, 15085, 15705, 15986, 16092, 16623, 16800, 17027, 17182, 17417, 17543, 17680, 17875, 17938, 18506, 18525, 18567, 18769, 18857, 18946, 19126, 19372, 19404, 19499, 19523, 19960, 20098, 20441, 20454, 20716, 20926, 21471, 21569, 21607, 21858, 21936, 22014, 22173, 22214, 22322, 22384, 22488, 22714, 22908, 22973, 23150, 25243, 26083, 28247, 29104, 152363, 153351, 162870]

ir_20A = [13910, 14496, 16623, 17543, 17875, 19372, 20098, 20441, 21471, 21569, 21607, 21858, 21936, 22173, 22214, 22322, 22384, 22714, 22908, 22973, 23150, 25243, 162870]

print (len(ir_all))

ir_04B = [14358, 15085, 15705, 15986, 16092, 16800, 17027, 17182, 17417, 17680, 17938, 18506, 18525, 18567, 18769, 18857, 18946, 19126, 19404, 19499, 19523, 19960, 20454, 20716, 20926, 22014, 22488, 26083, 28247, 29104, 152363, 153351]
print (len(ir_04B))


#ir_20A = [x for x in ir_all if x not in ir_04B]

print(len(ir_20A))
print("IDs in 20A")
print (ir_20A)

ir_1_1_ids = [13910, 17875, 21471, 21858, 23150] 
ir_1_2_ids = [19372, 20441, 21569, 21607, 21936, 22173, 162870]
ir_1_3_ids = [17543, 22384, 22908, 22973]

ir_1_4_ids = [16623, 20098, 22214, 25243]
ir_1_5_ids = [22714,14496]
ir_1_6_ids = [22322]
#ir_1_remaining = [x for x in ir_20A if x not in ir_1_1_ids  and x not in ir_1_2_ids and x not in ir_1_3_ids and x not in ir_1_4_ids and x not in ir_1_5_ids]

ir_1_remaining = [x for x in ir_20A if x not in ir_1_1_ids and x not in ir_1_2_ids and x not in ir_1_3_ids and x not in ir_1_4_ids and x not in ir_1_5_ids and x not in ir_1_6_ids]
print("IDs IR remaining:", ir_1_remaining)

#did I miss anything: 
all_groups = (
    set(ir_1_1_ids)
    | set(ir_1_2_ids)
    | set(ir_1_3_ids)
    | set(ir_1_4_ids)
    | set(ir_1_5_ids)
    | set(ir_1_6_ids)
  
)


ir_set_20A = set(ir_20A)

missing = ir_set_20A - all_groups
extra = all_groups - ir_set_20A

print("Missing from groups:", missing)
print("Extra in groups:", extra)


#did I duplicate any neuron?
from collections import Counter

all_ids_list = (
    ir_1_1_ids
    + ir_1_2_ids
    + ir_1_3_ids
    + ir_1_4_ids
    + ir_1_5_ids
    + ir_1_6_ids
  
)

counts = Counter(all_ids_list)

duplicates = {k: v for k, v in counts.items() if v > 1}
print("Duplicates across groups:", duplicates)




55
32
23
IDs in 20A
[13910, 14496, 16623, 17543, 17875, 19372, 20098, 20441, 21471, 21569, 21607, 21858, 21936, 22173, 22214, 22322, 22384, 22714, 22908, 22973, 23150, 25243, 162870]
IDs IR remaining: []
Missing from groups: set()
Extra in groups: set()
Duplicates across groups: {}
